# ⚡ PySpark: Junior vs Senior
Mismo resultado, pero el senior
escribe código que ESCALA a
millones de filas.

In [1]:
from pyspark.sql import (
    SparkSession, functions as F,
    Window)

spark = (
    SparkSession.builder
    .appName('junior_vs_senior')
    .getOrCreate()
)

# Ventas: varias por usuario
ventas = spark.createDataFrame([
  ('u1','2025-01-05','Madrid',120),
  ('u1','2025-03-10','Madrid',80),
  ('u2','2025-02-01','Barna',200),
  ('u2','2025-02-15','Barna',50),
  ('u3','2025-01-20','Sevilla',300),
], ['usuario','fecha','ciudad','importe'])

# Tabla pequeña de planes
planes = spark.createDataFrame([
  ('u1','Premium'), ('u2','Free'),
  ('u3','Premium'),
], ['usuario','plan'])

### 💼 Quédate con la ÚLTIMA
### compra de cada usuario

In [2]:
from pyspark.sql import (
    SparkSession, functions as F,
    Window)

spark = (
    SparkSession.builder
    .appName('junior_vs_senior')
    .getOrCreate()
)

# Ventas: varias por usuario
ventas = spark.createDataFrame([
  ('u1','2025-01-05','Madrid',120),
  ('u1','2025-03-10','Madrid',80),
  ('u2','2025-02-01','Barna',200),
  ('u2','2025-02-15','Barna',50),
  ('u3','2025-01-20','Sevilla',300),
], ['usuario','fecha','ciudad','importe'])

ventas.show()

+-------+----------+-------+-------+
|usuario|     fecha| ciudad|importe|
+-------+----------+-------+-------+
|     u1|2025-01-05| Madrid|    120|
|     u1|2025-03-10| Madrid|     80|
|     u2|2025-02-01|  Barna|    200|
|     u2|2025-02-15|  Barna|     50|
|     u3|2025-01-20|Sevilla|    300|
+-------+----------+-------+-------+



In [3]:
# JUNIOR ❌ groupBy + join de vuelta
ult = (
  ventas
  .groupBy('usuario')
  .agg(F.max('fecha').alias('fecha'))
)

ventas.join(
  ult, ['usuario','fecha']).show()

+-------+----------+-------+-------+
|usuario|     fecha| ciudad|importe|
+-------+----------+-------+-------+
|     u1|2025-03-10| Madrid|     80|
|     u2|2025-02-15|  Barna|     50|
|     u3|2025-01-20|Sevilla|    300|
+-------+----------+-------+-------+



In [4]:
# SENIOR ✅ Window (una sola pasada)
w = (
  Window
  .partitionBy('usuario')
  .orderBy(F.col('fecha').desc())
)

(ventas
 .withColumn('rk', F.row_number().over(w))
 .filter(F.col('rk') == 1)
 .drop('rk')
 .show())

+-------+----------+-------+-------+
|usuario|     fecha| ciudad|importe|
+-------+----------+-------+-------+
|     u1|2025-03-10| Madrid|     80|
|     u2|2025-02-15|  Barna|     50|
|     u3|2025-01-20|Sevilla|    300|
+-------+----------+-------+-------+



### 💼 Une ventas con su plan

La tabla de planes es diminuta.
El junior hace un join normal
(mueve la tabla grande por la red).
El senior la DIFUNDE (broadcast).

In [5]:
# JUNIOR ❌ join normal → SortMergeJoin
(ventas
 .join(planes, 'usuario')
 .explain())

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [usuario#6, fecha#7, ciudad#8, importe#9L, plan#5]
   +- SortMergeJoin [usuario#6], [usuario#4], Inner
      :- Sort [usuario#6 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(usuario#6, 200), ENSURE_REQUIREMENTS, [plan_id=330]
      :     +- Filter isnotnull(usuario#6)
      :        +- Scan ExistingRDD[usuario#6,fecha#7,ciudad#8,importe#9L]
      +- Sort [usuario#4 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(usuario#4, 200), ENSURE_REQUIREMENTS, [plan_id=331]
            +- Filter isnotnull(usuario#4)
               +- Scan ExistingRDD[usuario#4,plan#5]




In [6]:
# SENIOR ✅ broadcast → BroadcastHashJoin
(ventas
 .join(F.broadcast(planes), 'usuario')
 .explain())

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [usuario#6, fecha#7, ciudad#8, importe#9L, plan#5]
   +- BroadcastHashJoin [usuario#6], [usuario#4], Inner, BuildRight, false, false
      :- Filter isnotnull(usuario#6)
      :  +- Scan ExistingRDD[usuario#6,fecha#7,ciudad#8,importe#9L]
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=360]
         +- Filter isnotnull(usuario#4)
            +- Scan ExistingRDD[usuario#4,plan#5]




### 💼 ¿Hay alguna compra
### de más de 250€?

El junior cuenta TODAS y compara.
El senior para en cuanto encuentra
la primera.

In [7]:
# JUNIOR ❌ recorre todo para contar
caras = ventas.filter(
  F.col('importe') > 250)

print(caras.count() > 0)

True


In [8]:
# SENIOR ✅ para al encontrar 1
caras = ventas.filter(
  F.col('importe') > 250)

print(len(caras.take(1)) > 0)

True


In [9]:
spark.stop()